# Lead.AI Fraud Shield — Verified Fraud Benchmark Pipeline

This notebook is a reproducible research and portfolio demonstration. It does **not** claim production fraud-detection performance.

The attached 105-row Lead.AI CSV is inspected as a schema/smoke-test asset. Model selection and final metrics use a larger probabilistic synthetic generator, a validation-selected threshold, and an untouched test split. The incompatible sample dataset is not merged blindly.


In [ ]:
from pathlib import Path
import json
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, matthews_corrcoef, precision_recall_curve,
    precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
print("Environment ready; seed =", SEED)


In [ ]:
# Inspect the attached small dataset without treating it as a production benchmark.
candidates = [
    Path("/kaggle/input/lead-ai-fraud-detection-table-data/train.csv"),
    Path("../fraud-detection-table-data/train.csv"),
    Path("../../datasets/fraud-detection-Table-data/train.csv"),
]
small_path = next((p for p in candidates if p.exists()), None)
if small_path is None:
    print("Attached 105-row CSV not found; continuing with the reproducible generator.")
else:
    small_df = pd.read_csv(small_path)
    print("Smoke-test dataset:", small_path)
    print("Shape:", small_df.shape)
    print("Class counts:", small_df["is_fraud"].value_counts().sort_index().to_dict())
    display(small_df.head())


In [ ]:
def generate_synthetic_transactions(n_rows=5000, seed=42):
    if n_rows < 200:
        raise ValueError("n_rows must be at least 200")
    rng = np.random.default_rng(seed)
    amount = np.clip(rng.lognormal(5.25, 1.05, n_rows), 1, 10000)
    transaction_hour = rng.integers(0, 24, n_rows)
    merchant_risk_score = rng.beta(2.0, 4.0, n_rows)
    customer_age_days = np.clip(rng.gamma(2.2, 260, n_rows), 1, 3000).astype(int)
    device_trust_score = rng.beta(4.5, 2.0, n_rows)
    location_risk_score = rng.beta(1.8, 4.2, n_rows)
    velocity_24h = np.clip(rng.poisson(3.5, n_rows) + 1, 1, 40)
    previous_chargebacks = np.clip(rng.poisson(0.25, n_rows), 0, 8)
    payment_method_risk = rng.beta(2.0, 4.0, n_rows)
    night = np.isin(transaction_hour, [23, 0, 1, 2, 3, 4]).astype(float)
    linear_risk = (
        -7.0 + 0.00135 * amount + 5.40 * merchant_risk_score
        - 5.40 * device_trust_score + 5.40 * location_risk_score
        + 0.324 * velocity_24h + 1.98 * previous_chargebacks
        + 3.96 * payment_method_risk + 1.98 * night
        - 0.00216 * customer_age_days + rng.normal(0, 0.20, n_rows)
    )
    fraud_probability = 1 / (1 + np.exp(-linear_risk))
    is_fraud = rng.binomial(1, fraud_probability)
    return pd.DataFrame({
        "amount": amount,
        "transaction_hour": transaction_hour,
        "merchant_risk_score": merchant_risk_score,
        "customer_age_days": customer_age_days,
        "device_trust_score": device_trust_score,
        "location_risk_score": location_risk_score,
        "velocity_24h": velocity_24h,
        "previous_chargebacks": previous_chargebacks,
        "payment_method_risk": payment_method_risk,
        "is_fraud": is_fraud,
    })

frame = generate_synthetic_transactions(5000, SEED)
X = frame.drop(columns="is_fraud")
y = frame["is_fraud"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print({"train": len(X_train), "validation": len(X_val), "test": len(X_test),
       "test_positive_rate": float(y_test.mean())})


In [ ]:
candidates = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=350, min_samples_leaf=2, class_weight="balanced_subsample",
        n_jobs=-1, random_state=SEED
    ),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
for name, estimator in candidates.items():
    oof = cross_val_predict(estimator, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    rows.append({
        "model": name,
        "cv_pr_auc": average_precision_score(y_train, oof),
        "cv_roc_auc": roc_auc_score(y_train, oof),
    })
selection = pd.DataFrame(rows).sort_values(["cv_pr_auc", "cv_roc_auc"], ascending=False)
display(selection)
selected_name = selection.iloc[0]["model"]
model = candidates[selected_name].fit(X_train, y_train)
print("Selected:", selected_name)


In [ ]:
def choose_threshold(y_true, probability, minimum_recall=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, probability)
    valid = [
        (precision[i], thresholds[i])
        for i in range(len(thresholds)) if recall[i] >= minimum_recall
    ]
    return float(max(valid, key=lambda item: (item[0], item[1]))[1]) if valid else 0.5

val_probability = model.predict_proba(X_val)[:, 1]
threshold = choose_threshold(y_val, val_probability, 0.80)
print("Validation-selected threshold:", threshold)


In [ ]:
start = time.perf_counter()
test_probability = model.predict_proba(X_test)[:, 1]
latency_ms_per_row = (time.perf_counter() - start) * 1000 / len(X_test)
test_prediction = (test_probability >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_prediction, labels=[0, 1]).ravel()
metrics = {
    "selected_model": selected_name,
    "rows": int(len(X_test)),
    "positive_rate": float(y_test.mean()),
    "threshold": threshold,
    "pr_auc": average_precision_score(y_test, test_probability),
    "roc_auc": roc_auc_score(y_test, test_probability),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_prediction),
    "precision": precision_score(y_test, test_prediction, zero_division=0),
    "recall": recall_score(y_test, test_prediction, zero_division=0),
    "f1": f1_score(y_test, test_prediction, zero_division=0),
    "mcc": matthews_corrcoef(y_test, test_prediction),
    "brier_score": brier_score_loss(y_test, test_probability),
    "latency_ms_per_row": latency_ms_per_row,
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
}
display(pd.DataFrame([metrics]).drop(columns="confusion_matrix").T.rename(columns={0: "value"}))
print("Confusion matrix:", metrics["confusion_matrix"])


In [ ]:
if selected_name == "logistic_regression":
    importance = pd.DataFrame({
        "feature": X.columns,
        "signed_standardized_coefficient": model.named_steps["model"].coef_[0],
    })
    importance["absolute_importance"] = importance["signed_standardized_coefficient"].abs()
else:
    importance = pd.DataFrame({
        "feature": X.columns,
        "absolute_importance": model.feature_importances_,
    })
display(importance.sort_values("absolute_importance", ascending=False))


In [ ]:
output_dir = Path("/kaggle/working/lead-ai-fraud-shield-artifacts")
output_dir.mkdir(parents=True, exist_ok=True)
bundle = {
    "estimator": model,
    "feature_names": list(X.columns),
    "threshold": threshold,
    "selected_model": selected_name,
    "metadata": {"seed": SEED, "source": "probabilistic synthetic generator"},
}
joblib.dump(bundle, output_dir / "model.joblib")
(output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
(output_dir / "feature_schema.json").write_text(json.dumps({"features": list(X.columns)}, indent=2))
print("Saved:", sorted(p.name for p in output_dir.iterdir()))


## Responsible-use conclusion

The notebook demonstrates reproducible engineering, not production readiness. Real deployment requires representative labeled data, institution-specific thresholds, calibration, drift and fairness monitoring, security review, compliance review, incident response, and human oversight. Do not use this baseline as the sole basis for accusations, account blocking, credit decisions, or law-enforcement reporting.
